1. Create a simple Python script using Keras to build an LSTM layer for predicting the next value in a sequence of daily step counts (e.g., [4000, 4200, 4300, 4100, 4500, ...]) like those tracked by a fitness app.

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

steps = np.array([4000, 4200, 4300, 4100, 4500, 4700, 4900, 5000])

X = []
y = []

for i in range(len(steps) - 3):
    X.append(steps[i:i+3])
    y.append(steps[i+3])

X = np.array(X)
y = np.array(y)

X = X.reshape((X.shape[0], X.shape[1], 1))

model = Sequential([
    LSTM(32, input_shape=(3, 1)),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

model.fit(X, y, epochs=50, verbose=1)

test_sequence = np.array([[4700, 4900, 5000]])
test_sequence = test_sequence.reshape((1, 3, 1))

prediction = model.predict(test_sequence)

print("Predicted next day's steps:", prediction[0][0])

c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 21648268.0000
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - loss: 21647788.0000
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - loss: 21647692.0000
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - loss: 21647524.0000
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 21646580.0000
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 21645040.0000
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 21644704.0000
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 21644618.0000
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 21644514.0000
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - loss: 21644170.0000
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - loss: 21643452.0000
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 21643180.0000
Epoch 13/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 21643090.0000
Epoch 14/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - loss: 21643008.0000
E

2. Write a short code snippet to demonstrate the vanishing gradient problem in a basic RNN by training it on a sequence of 0s and 1s, then print the gradients after each epoch.<br><br><em><strong>Hint:</strong> Use TensorFlow or PyTorch and compare the gradients at early vs late time steps.</em>


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)

class SimpleRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=8, output_size=1):
        super(SimpleRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, hidden = self.rnn(x)
        out = self.fc(out[:, -1, :])  # Use final time step
        return out

seq_length = 30
X = torch.randint(0, 2, (100, seq_length, 1)).float()

y = (X.sum(dim=1) % 2).float()

model = SimpleRNN()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

print("Training and monitoring gradients...\n")

for epoch in range(10):
    optimizer.zero_grad()

    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()

    grad_norm = model.rnn.weight_hh_l0.grad.norm().item()

    print(
        f"Epoch {epoch+1:2d} | "
        f"Loss: {loss.item():.4f} | "
        f"Recurrent Gradient Norm: {grad_norm:.8f}"
    )

    optimizer.step()

Training and monitoring gradients...

Epoch  1 | Loss: 0.6332 | Recurrent Gradient Norm: 0.61815637
Epoch  2 | Loss: 0.5932 | Recurrent Gradient Norm: 0.58305806
Epoch  3 | Loss: 0.5575 | Recurrent Gradient Norm: 0.55140692
Epoch  4 | Loss: 0.5256 | Recurrent Gradient Norm: 0.52257067
Epoch  5 | Loss: 0.4969 | Recurrent Gradient Norm: 0.49604648
Epoch  6 | Loss: 0.4712 | Recurrent Gradient Norm: 0.47143689
Epoch  7 | Loss: 0.4481 | Recurrent Gradient Norm: 0.44842902
Epoch  8 | Loss: 0.4273 | Recurrent Gradient Norm: 0.42677808
Epoch  9 | Loss: 0.4086 | Recurrent Gradient Norm: 0.40629336
Epoch 10 | Loss: 0.3919 | Recurrent Gradient Norm: 0.38682681


3. Draw and label the architecture of a single LSTM cell, clearly marking the cell state, input gate, forget gate, and output gate. Annotate each gate with a one-line explanation of its function.


                 ┌──────────────────────────────┐
h(t-1) ────────▶ │                              │
x(t)   ────────▶ │   LSTM CELL                 │
                 │                              │
                 │  Forget Gate (fₜ)           │
                 │  σ(Wf·[hₜ₋₁, xₜ])            │
                 │  → decides what to erase     │
                 │                              │
                 │  Input Gate (iₜ)            │
                 │  σ(Wi·[hₜ₋₁, xₜ])            │
                 │  → controls new info flow    │
                 │                              │
                 │  Candidate State (Ĉₜ)       │
                 │  tanh(Wc·[hₜ₋₁, xₜ])        │
                 │  → creates new memory        │
                 │                              │
                 │  Cell State (Cₜ) ────────────┼────────▶ Cₜ (memory carried forward)
                 │  Cₜ = fₜ*Cₜ₋₁ + iₜ*Ĉₜ        │
                 │  → long-term memory storage  │
                 │                              │
                 │  Output Gate (oₜ)           │
                 │  σ(Wo·[hₜ₋₁, xₜ])            │
                 │  → controls output exposure  │
                 │                              │
                 │  Hidden State (hₜ)           │
                 │  hₜ = oₜ * tanh(Cₜ)             │
                 │  → final output of step      │
                 └─────────────┬────────────────┘
                               │
                               ▼
                             h(t)

4. Given a sample input sequence representing a user's recent food orders on Zomato (e.g., ['pizza', 'burger', 'pasta', 'pizza']), manually walk through the LSTM cell's information flow for two time steps, showing what information the forget, input, and output gates would keep or discard at each step.<br><br><em><strong>Hint:</strong> Use simple binary values (0 or 1) for gate outputs to illustrate the concept.</em>

Let’s walk through a manual LSTM information flow using your sequence:

Input sequence:
['pizza', 'burger', 'pasta', 'pizza']

We’ll simulate 2 time steps and use binary gate values (0 = blocked, 1 = allowed) to make the flow easy to understand.

 Assumption for Simplification

We track a single “memory topic”:

“User likes pizza”

So the cell state (Cₜ) represents how strongly the model remembers “pizza preference”.

 TIME STEP 1 → Input: “pizza”
Initial state:
C₀ = 0 (no memory yet)
1. Forget Gate (f₁)

 What to keep from old memory?

f₁ = 0
Reason: No useful past memory exists yet

✔ Effect:

C₀ is ignored
2. Input Gate (i₁)

 Should we store new info (“pizza”)?

i₁ = 1
Reason: “pizza” is important signal

✔ Effect:

New info is written into memory
3. Candidate Memory (Ĉ₁)
Ĉ₁ = “pizza”
4. Cell State Update
C
1
	​

=f
1
	​

×C
0
	​

+i
1
	​

×
C
^
1
	​

C
1
	​

=0×0+1×pizza=pizza

✔ Memory now:

C₁ =  (strong pizza preference stored)

5. Output Gate (o₁)
o₁ = 1
Reason: We can output current preference

✔ Hidden state:

h₁ = “pizza preference”
 TIME STEP 2 → Input: “burger”

Now model receives:

new input = “burger”
previous memory = “pizza”

1. Forget Gate (f₂)

 Should we keep old memory?

f₂ = 0
Reason: “burger” is different, so we suppress old pizza preference

✔ Effect:

pizza memory is discarded
2. Input Gate (i₂)

 Should we store burger?

i₂ = 1
Reason: new valid input

✔ Candidate:

Ĉ₂ = “burger”
3. Cell State Update
C
2
	​

=f
2
	​

×C
1
	​

+i
2
	​

×
C
^
2
	​

C
2
	​

=0×pizza+1×burger=burger

✔ Memory now:

C₂ = (pizza memory replaced)

4. Output Gate (o₂)
o₂ = 1
Reason: model outputs current preference

✔ Hidden state:

h₂ = “burger preference”
Summary Table (2 Steps)
Time	Input	Forget Gate	Input Gate	Cell State (Memory)	Output
t₁	pizza	0	1	pizza	pizza preference
t₂	burger	0	1	burger	burger preference
 Key Insight
Forget gate = decides old memory survival
Input gate = decides new memory writing
Output gate = decides what is shown externally
Cell state = long-term memory (like a running preference log)

If you want, I can also extend this to:

full 4-step sequence (pizza → burger → pasta → pizza)
or show how LSTM would not fully forget pizza in real scenarios (soft values instead of 0/1)
or convert this into a diagram for exam writing

5. Write a short explanation (3-4 lines) of what 'unrolling an LSTM through time' means, using an example from a music playlist app that predicts the next song a user might play.

“Unrolling an LSTM through time” means visualizing how the same LSTM cell is applied repeatedly across a sequence, one time step after another, while sharing the same weights. For example, in a music playlist app, each song a user plays (like “rock → pop → jazz”) is fed into the LSTM step by step. At each step, the model updates its memory and predicts the next likely song based on both the current input and past listening history. Unrolling simply shows this repeated process across time as a chain of connected LSTM cells.